# 💊 PrescriBot — India-Focused Drug Knowledge Base Builder

Builds the complete knowledge base for PrescriBot:

| Step | Output |
|------|--------|
| Install deps | — |
| Imports + config | — |
| Indian brand mapping (150 drugs) | curated dataset |
| OpenFDA fetch | 500 drug records |
| Merge datasets | combined records |
| Qdrant vector store (Hybrid RAG) | `./qdrant_data/` |
| SQLite database | `prescribot.db` |

> ⚠️ Run once before launching the Streamlit app.

---

## Step 1 — Install Dependencies

In [ ]:
!pip install requests qdrant-client sentence-transformers rank-bm25 tqdm

## Step 2 — Imports & Configuration

Key settings:
- `OPENFDA_PAGES` — pages to fetch (100 drugs/page). Default 5 = 500 drugs.
- `DENSE_MODEL` — `BAAI/bge-small-en-v1.5` is lightweight and good for medical text.
- `DB_PATH` — SQLite file location.

In [ ]:
import os
import re
import json
import time
import sqlite3
import requests
from tqdm import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, SparseVector,
    SparseVectorParams, SparseIndexParams
)
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
QDRANT_PATH      = "./qdrant_data"          # local Qdrant storage
COLLECTION_NAME  = "drug_knowledge"
DB_PATH          = "prescribot.db"
DENSE_MODEL      = "BAAI/bge-small-en-v1.5" # lightweight, good for medical text
VECTOR_SIZE      = 384                       # matches bge-small output
OPENFDA_LIMIT    = 100                       # records per API page (max 1000)
OPENFDA_PAGES    = 5                         # fetch 5 pages = 500 drugs total
OPENFDA_SLEEP    = 0.5                       # seconds between API calls (be polite)

print("✅ Imports and config ready")

## Step 3 — Indian Brand → Generic Drug Mapping

OpenFDA is US-focused and doesn't know that **Dolo 650 = Paracetamol** or **Pan-D = Pantoprazole + Domperidone**. This curated list of 150 Indian brand names bridges that gap.

In [ ]:
INDIAN_DRUGS = [
    # Painkillers / Antipyretics
    ("Dolo 650",        "Paracetamol",                          "Analgesic/Antipyretic"),
    ("Crocin",          "Paracetamol",                          "Analgesic/Antipyretic"),
    ("Calpol",          "Paracetamol",                          "Analgesic/Antipyretic"),
    ("Combiflam",       "Ibuprofen + Paracetamol",              "Analgesic/Antipyretic"),
    ("Brufen",          "Ibuprofen",                            "NSAID"),
    ("Zerodol",         "Aceclofenac",                          "NSAID"),
    ("Nise",            "Nimesulide",                           "NSAID"),
    ("Voveran",         "Diclofenac",                           "NSAID"),
    ("Flexon",          "Ibuprofen + Paracetamol",              "NSAID"),
    ("Meftal Spas",     "Mefenamic Acid + Dicyclomine",         "Antispasmodic/Analgesic"),

    # Antibiotics
    ("Azithral",        "Azithromycin",                         "Antibiotic"),
    ("Zithromax",       "Azithromycin",                         "Antibiotic"),
    ("Ciplox",          "Ciprofloxacin",                        "Antibiotic"),
    ("Taxim-O",         "Cefixime",                             "Antibiotic"),
    ("Moxikind-CV",     "Amoxicillin + Clavulanic Acid",        "Antibiotic"),
    ("Augmentin",       "Amoxicillin + Clavulanic Acid",        "Antibiotic"),
    ("Levoflox",        "Levofloxacin",                         "Antibiotic"),
    ("Zifi",            "Cefixime",                             "Antibiotic"),
    ("Oflox",           "Ofloxacin",                            "Antibiotic"),
    ("Monocef",         "Ceftriaxone",                          "Antibiotic"),
    ("Clavam",          "Amoxicillin + Clavulanic Acid",        "Antibiotic"),
    ("Sporidex",        "Cephalexin",                           "Antibiotic"),

    # Antifungals
    ("Flucos",          "Fluconazole",                          "Antifungal"),
    ("Forcan",          "Fluconazole",                          "Antifungal"),
    ("Canditral",       "Itraconazole",                         "Antifungal"),
    ("Zocon",           "Fluconazole",                          "Antifungal"),

    # Antivirals / Antiparasitics
    ("Valcivir",        "Valacyclovir",                         "Antiviral"),
    ("Acivir",          "Acyclovir",                            "Antiviral"),
    ("Zentel",          "Albendazole",                          "Antiparasitic"),
    ("Bandy",           "Albendazole",                          "Antiparasitic"),
    ("Mebex",           "Mebendazole",                          "Antiparasitic"),

    # Gastrointestinal
    ("Pan-D",           "Pantoprazole + Domperidone",           "GI / PPI"),
    ("Pantocid",        "Pantoprazole",                         "Proton Pump Inhibitor"),
    ("Nexpro",          "Esomeprazole",                         "Proton Pump Inhibitor"),
    ("Razo",            "Rabeprazole",                          "Proton Pump Inhibitor"),
    ("Omez",            "Omeprazole",                           "Proton Pump Inhibitor"),
    ("Rantac",          "Ranitidine",                           "H2 Blocker"),
    ("Zantac",          "Ranitidine",                           "H2 Blocker"),
    ("Cremaffin",       "Liquid Paraffin + Milk of Magnesia",   "Laxative"),
    ("Dulcolax",        "Bisacodyl",                            "Laxative"),
    ("Enterogermina",   "Bacillus clausii",                     "Probiotic"),
    ("Aristozyme",      "Digestive Enzymes",                    "Digestive"),
    ("Cyclopam",        "Dicyclomine + Paracetamol",            "Antispasmodic"),

    # Diabetes
    ("Glucophage",      "Metformin",                            "Antidiabetic"),
    ("Obimet",          "Metformin",                            "Antidiabetic"),
    ("Glycomet",        "Metformin",                            "Antidiabetic"),
    ("Amaryl",          "Glimepiride",                          "Antidiabetic"),
    ("Zoryl",           "Glimepiride",                          "Antidiabetic"),
    ("Janumet",         "Sitagliptin + Metformin",              "Antidiabetic"),
    ("Galvus Met",      "Vildagliptin + Metformin",             "Antidiabetic"),
    ("Jardiance",       "Empagliflozin",                        "Antidiabetic"),
    ("Forxiga",         "Dapagliflozin",                        "Antidiabetic"),
    ("Trajenta",        "Linagliptin",                          "Antidiabetic"),
    ("Victoza",         "Liraglutide",                          "Antidiabetic"),
    ("Insulin Glargine","Insulin Glargine",                     "Antidiabetic / Insulin"),
    ("Mixtard",         "Insulin (Biphasic)",                   "Antidiabetic / Insulin"),

    # Cardiovascular / Hypertension
    ("Atorva",          "Atorvastatin",                         "Statin"),
    ("Atorlip",         "Atorvastatin",                         "Statin"),
    ("Lipitor",         "Atorvastatin",                         "Statin"),
    ("Storvas",         "Atorvastatin",                         "Statin"),
    ("Rosuvas",         "Rosuvastatin",                         "Statin"),
    ("Crestor",         "Rosuvastatin",                         "Statin"),
    ("Ecosprin",        "Aspirin",                              "Antiplatelet"),
    ("Clopilet",        "Clopidogrel",                          "Antiplatelet"),
    ("Plavix",          "Clopidogrel",                          "Antiplatelet"),
    ("Telma",           "Telmisartan",                          "ARB / Antihypertensive"),
    ("Telvas",          "Telmisartan",                          "ARB / Antihypertensive"),
    ("Amlodac",         "Amlodipine",                           "Calcium Channel Blocker"),
    ("Amlip",           "Amlodipine",                           "Calcium Channel Blocker"),
    ("Norvasc",         "Amlodipine",                           "Calcium Channel Blocker"),
    ("Olmezest",        "Olmesartan",                           "ARB / Antihypertensive"),
    ("Nebicard",        "Nebivolol",                            "Beta Blocker"),
    ("Ramipres",        "Ramipril",                             "ACE Inhibitor"),
    ("Cardace",         "Ramipril",                             "ACE Inhibitor"),
    ("Metolar",         "Metoprolol",                           "Beta Blocker"),
    ("Betaloc",         "Metoprolol",                           "Beta Blocker"),
    ("Aten",            "Atenolol",                             "Beta Blocker"),
    ("Stamlo",          "Amlodipine",                           "Calcium Channel Blocker"),

    # Thyroid
    ("Eltroxin",        "Levothyroxine",                        "Thyroid Hormone"),
    ("Thyronorm",       "Levothyroxine",                        "Thyroid Hormone"),
    ("Thyrox",          "Levothyroxine",                        "Thyroid Hormone"),

    # Respiratory / Allergy
    ("Asthalin",        "Salbutamol",                           "Bronchodilator"),
    ("Deriphyllin",     "Theophylline + Etofylline",            "Bronchodilator"),
    ("Montair",         "Montelukast",                          "Leukotriene Antagonist"),
    ("Singulair",       "Montelukast",                          "Leukotriene Antagonist"),
    ("Foracort",        "Formoterol + Budesonide",              "Inhaled Corticosteroid"),
    ("Seroflo",         "Salmeterol + Fluticasone",             "Inhaled Corticosteroid"),
    ("Allegra",         "Fexofenadine",                         "Antihistamine"),
    ("Cetrizine",       "Cetirizine",                           "Antihistamine"),
    ("Levocet",         "Levocetirizine",                       "Antihistamine"),
    ("Atarax",          "Hydroxyzine",                          "Antihistamine"),

    # Vitamins / Supplements
    ("Shelcal",         "Calcium + Vitamin D3",                 "Supplement"),
    ("Calcirol",        "Cholecalciferol (Vitamin D3)",         "Supplement"),
    ("Becosules",       "Vitamin B Complex",                    "Supplement"),
    ("Neurobion",       "Vitamin B1+B6+B12",                    "Supplement"),
    ("Zincovit",        "Zinc + Multivitamins",                 "Supplement"),
    ("Supradyn",        "Multivitamin + Minerals",              "Supplement"),
    ("Folvite",         "Folic Acid",                           "Supplement"),
    ("Ferrous Sulfate", "Ferrous Sulfate",                      "Iron Supplement"),
    ("Dexorange",       "Iron + Folic Acid + Vitamin B12",      "Supplement"),

    # Steroids / Anti-inflammatory
    ("Medrol",          "Methylprednisolone",                   "Corticosteroid"),
    ("Wysolone",        "Prednisolone",                         "Corticosteroid"),
    ("Defcort",         "Deflazacort",                          "Corticosteroid"),
    ("Betnesol",        "Betamethasone",                        "Corticosteroid"),

    # Neurological / Psychiatric
    ("Pregabalin",      "Pregabalin",                           "Neuropathic Pain"),
    ("Lyrica",          "Pregabalin",                           "Neuropathic Pain"),
    ("Gabapin",         "Gabapentin",                           "Neuropathic Pain"),
    ("Amitone",         "Amitriptyline",                        "Antidepressant"),
    ("Tryptomer",       "Amitriptyline",                        "Antidepressant"),
    ("Nexito",          "Escitalopram",                         "Antidepressant (SSRI)"),
    ("Stalopam",        "Escitalopram",                         "Antidepressant (SSRI)"),
    ("Restyl",          "Alprazolam",                           "Anxiolytic / Benzodiazepine"),
    ("Alprax",          "Alprazolam",                           "Anxiolytic / Benzodiazepine"),
    ("Lonazep",         "Clonazepam",                           "Anticonvulsant"),
    ("Clonafit",        "Clonazepam",                           "Anticonvulsant"),
    ("Epsolin",         "Phenytoin",                            "Anticonvulsant"),
    ("Topamate",        "Topiramate",                           "Anticonvulsant"),

    # Urology / Kidney
    ("Urimax",          "Tamsulosin",                           "Alpha Blocker / Urology"),
    ("Veltam",          "Tamsulosin",                           "Alpha Blocker / Urology"),
    ("Cystone",         "Herbal (Didymocarpus pedicellata)",    "Urinary / Herbal"),

    # Skin
    ("Betnovate",       "Betamethasone",                        "Topical Corticosteroid"),
    ("Candid-B",        "Clotrimazole + Beclomethasone",        "Antifungal + Steroid"),
    ("Terbicip",        "Terbinafine",                          "Antifungal"),
    ("Retino-A",        "Tretinoin",                            "Retinoid / Dermatology"),
    ("Elocon",          "Mometasone",                           "Topical Corticosteroid"),

    # Women's Health
    ("Primolut-N",      "Norethisterone",                       "Progestin"),
    ("Ovral-L",         "Levonorgestrel + Ethinyl Estradiol",   "Oral Contraceptive"),
    ("Femilon",         "Desogestrel + Ethinyl Estradiol",      "Oral Contraceptive"),
    ("Duphaston",       "Dydrogesterone",                       "Progestin"),
    ("Susten",          "Progesterone",                         "Progestin"),

    # Anti-nausea / Anti-vertigo
    ("Stemetil",        "Prochlorperazine",                     "Antiemetic"),
    ("Emeset",          "Ondansetron",                          "Antiemetic"),
    ("Ondem",           "Ondansetron",                          "Antiemetic"),
    ("Vertin",          "Betahistine",                          "Anti-vertigo"),

    # Miscellaneous
    ("Unienzyme",       "Fungal Diastase + Papain",             "Digestive Enzyme"),
    ("Liv 52",          "Herbal Hepatoprotective",              "Liver / Herbal"),
    ("Silymarin",       "Silymarin",                            "Liver / Herbal"),
    ("Glucon-D",        "Glucose Powder",                       "Energy Supplement"),
    ("ORS",             "Oral Rehydration Salts",               "Rehydration"),
    ("Electral",        "ORS (WHO Formula)",                    "Rehydration"),
]

print(f"✅ Loaded {len(INDIAN_DRUGS)} Indian brand → generic mappings")

## Step 4 — Fetch Drug Data from OpenFDA API

Fetches FDA-approved drug labels: indications, side effects, warnings, dosage, and interactions. Free — no API key needed for under 1000 requests/day.

In [ ]:
def fetch_openfda_drugs(pages: int = OPENFDA_PAGES, limit: int = OPENFDA_LIMIT) -> list[dict]:
    """
    Fetch drug label records from OpenFDA and extract the fields we care about.
    Returns a list of clean drug dicts.
    """
    base_url = "https://api.fda.gov/drug/label.json"
    drugs = []

    print(f"\n📥 Fetching {pages * limit} drug records from OpenFDA...")

    for page in range(pages):
        skip = page * limit
        params = {
            "search": "product_type:HUMAN+PRESCRIPTION+DRUG",
            "limit": limit,
            "skip": skip,
        }
        try:
            resp = requests.get(base_url, params=params, timeout=15)
            resp.raise_for_status()
            results = resp.json().get("results", [])

            for record in results:
                openfda = record.get("openfda", {})

                # Extract brand and generic names (lists in OpenFDA)
                brand_names  = openfda.get("brand_name", [])
                generic_names = openfda.get("generic_name", [])

                brand  = brand_names[0].title()  if brand_names  else "Unknown"
                generic = generic_names[0].title() if generic_names else brand

                # Core fields — all are lists in OpenFDA, take first element
                def first(key):
                    val = record.get(key, [])
                    return val[0].strip() if val else ""

                drug = {
                    "brand_name":        brand,
                    "generic_name":      generic,
                    "category":          "Prescription Drug",
                    "indications":       first("indications_and_usage") or first("purpose"),
                    "side_effects":      first("adverse_reactions") or first("side_effects"),
                    "warnings":          first("warnings") or first("warnings_and_cautions"),
                    "contraindications": first("contraindications"),
                    "dosage":            first("dosage_and_administration"),
                    "interactions":      first("drug_interactions"),
                    "source":            "OpenFDA",
                }

                # Skip records with no useful content
                if drug["indications"] or drug["side_effects"]:
                    drugs.append(drug)

            print(f"  Page {page+1}/{pages} — fetched {len(results)} records")
            time.sleep(OPENFDA_SLEEP)

        except Exception as e:
            print(f"  ⚠️  Page {page+1} failed: {e}")
            time.sleep(2)

    print(f"✅ OpenFDA: {len(drugs)} usable records fetched\n")
    return drugs


openfda_drugs = fetch_openfda_drugs()

## Step 5 — Merge OpenFDA Data with Indian Brand Names

For each Indian brand: find matching OpenFDA record by generic name, enrich it with clinical data. If no match found, create a stub record.

In [ ]:
def build_india_records(openfda_drugs: list[dict]) -> list[dict]:
    """
    For each Indian brand name, try to find matching OpenFDA record by generic name.
    If found, enrich it with OpenFDA data. If not, create a stub record.
    Returns combined list of all drug records to embed.
    """
    # Build lookup: generic_name (lower) → openfda record
    generic_lookup = {}
    for d in openfda_drugs:
        key = d["generic_name"].lower().strip()
        generic_lookup[key] = d

    combined = []

    for brand, generic, category in INDIAN_DRUGS:
        # Try exact match first, then partial match
        key = generic.lower().strip()
        match = generic_lookup.get(key)

        # Try partial match on first word of generic name
        if not match:
            first_word = key.split()[0]
            match = next(
                (v for k, v in generic_lookup.items() if first_word in k),
                None
            )

        if match:
            record = match.copy()
            record["brand_name"] = brand
            record["generic_name"] = generic
            record["category"] = category
            record["source"] = "OpenFDA + India Curated"
        else:
            # Stub — will be enriched by LLM at query time via RAG
            record = {
                "brand_name":        brand,
                "generic_name":      generic,
                "category":          category,
                "indications":       f"{brand} ({generic}) is used as a {category}.",
                "side_effects":      "",
                "warnings":          "",
                "contraindications": "",
                "dosage":            "",
                "interactions":      "",
                "source":            "India Curated",
            }

        combined.append(record)

    # Also add remaining OpenFDA records that weren't matched
    matched_generics = {r["generic_name"].lower() for r in combined}
    extras = [d for d in openfda_drugs
              if d["generic_name"].lower() not in matched_generics]
    combined.extend(extras[:200])   # cap at 200 extras to keep DB manageable

    print(f"✅ Combined dataset: {len(combined)} drug records\n")
    return combined


all_drugs = build_india_records(openfda_drugs)

# Quick preview
print("\n📋 Sample record (Dolo 650):")
sample = next((d for d in all_drugs if d["brand_name"] == "Dolo 650"), all_drugs[0])
for k, v in sample.items():
    print(f"  {k:20s}: {str(v)[:80]}")

## Step 6 — Build Qdrant Vector Store (Hybrid RAG)

Stores each drug as:
- **Dense vector** (BAAI/bge-small) — for semantic search
- **Sparse vector** (BM25) — for exact drug name keyword matching

> ⏳ Takes a few minutes — downloads the embedding model and encodes all records.

In [ ]:
def build_text(drug: dict) -> str:
    """Build the text blob we embed for each drug."""
    parts = [
        f"Drug: {drug['brand_name']} ({drug['generic_name']})",
        f"Category: {drug['category']}",
        f"Used for: {drug['indications'][:500]}"   if drug['indications']   else "",
        f"Side effects: {drug['side_effects'][:400]}" if drug['side_effects'] else "",
        f"Warnings: {drug['warnings'][:300]}"       if drug['warnings']      else "",
        f"Dosage: {drug['dosage'][:300]}"            if drug['dosage']        else "",
    ]
    return "\n".join(p for p in parts if p)


def tokenize(text: str) -> list[str]:
    return re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()


def build_sparse_vector(tokens: list[str], bm25: BM25Okapi, vocab: dict) -> SparseVector:
    """Convert BM25 token scores to a sparse vector for Qdrant."""
    scores = {}
    for token in set(tokens):
        idx = vocab.get(token)
        if idx is not None:
            idf = bm25.idf.get(token, 0)
            scores[idx] = float(idf)
    if not scores:
        return SparseVector(indices=[0], values=[0.0])
    indices, values = zip(*scores.items())
    return SparseVector(indices=list(indices), values=list(values))


def build_qdrant_collection(drugs: list[dict]):
    print("🔧 Loading embedding model...")
    model = SentenceTransformer(DENSE_MODEL)

    print("🔧 Building BM25 index for sparse vectors...")
    texts  = [build_text(d) for d in drugs]
    corpus = [tokenize(t) for t in texts]
    bm25   = BM25Okapi(corpus)

    # Vocabulary: token → integer index
    vocab = {token: idx for idx, token in
             enumerate(sorted({t for doc in corpus for t in doc}))}

    print("🔧 Connecting to Qdrant (local mode)...")
    client = QdrantClient(path=QDRANT_PATH)

    # Recreate collection
    if client.collection_exists(COLLECTION_NAME):
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
        },
        sparse_vectors_config={
            "sparse": SparseVectorParams(
                index=SparseIndexParams(on_disk=False)
            )
        }
    )

    print(f"📦 Embedding and uploading {len(drugs)} drug records...")
    batch_size = 32

    for i in tqdm(range(0, len(drugs), batch_size)):
        batch_drugs  = drugs[i:i+batch_size]
        batch_texts  = texts[i:i+batch_size]
        batch_corpus = corpus[i:i+batch_size]

        dense_vecs = model.encode(batch_texts, normalize_embeddings=True)

        points = []
        for j, (drug, text, tokens, dense) in enumerate(
            zip(batch_drugs, batch_texts, batch_corpus, dense_vecs)
        ):
            sparse = build_sparse_vector(tokens, bm25, vocab)
            points.append(PointStruct(
                id=i + j,
                vector={"dense": dense.tolist(), "sparse": sparse},
                payload={
                    "brand_name":        drug["brand_name"],
                    "generic_name":      drug["generic_name"],
                    "category":          drug["category"],
                    "indications":       drug["indications"][:800],
                    "side_effects":      drug["side_effects"][:600],
                    "warnings":          drug["warnings"][:500],
                    "contraindications": drug["contraindications"][:400],
                    "dosage":            drug["dosage"][:400],
                    "source":            drug["source"],
                    "text":              text,
                }
            ))

        client.upsert(collection_name=COLLECTION_NAME, points=points)

    # Save vocab for retrieval use
    with open("bm25_vocab.json", "w") as f:
        json.dump(vocab, f)

    print(f"✅ Qdrant collection '{COLLECTION_NAME}' built with {len(drugs)} records\n")
    return client


qdrant_client = build_qdrant_collection(all_drugs)

## Step 7 — Build SQLite Database

3 tables:

| Table | Purpose |
|-------|---------|
| `drugs` | Brand → generic + category |
| `interactions` | Drug-drug interaction pairs with severity |
| `dosage_ranges` | Safe min/max dose ranges |

In [ ]:
INTERACTIONS = [
    ("Warfarin",      "Aspirin",         "high",     "Increased bleeding risk. Avoid combination or monitor INR closely."),
    ("Warfarin",      "Ibuprofen",       "high",     "NSAIDs increase anticoagulant effect and GI bleeding risk."),
    ("Metformin",     "Alcohol",         "moderate", "Increased risk of lactic acidosis."),
    ("Metformin",     "Contrast Dye",    "high",     "Hold metformin 48 hrs before/after iodinated contrast."),
    ("Aspirin",       "Clopidogrel",     "moderate", "Dual antiplatelet — increased bleeding risk. Often intentional."),
    ("Atorvastatin",  "Clarithromycin",  "high",     "CYP3A4 inhibition raises statin levels → myopathy risk."),
    ("Atorvastatin",  "Erythromycin",    "moderate", "CYP3A4 inhibitor — monitor for muscle pain."),
    ("Amlodipine",    "Simvastatin",     "moderate", "Amlodipine raises simvastatin levels. Cap simvastatin at 20mg."),
    ("Ciprofloxacin", "Antacids",        "moderate", "Antacids reduce ciprofloxacin absorption. Take 2 hrs apart."),
    ("Levothyroxine", "Calcium",         "moderate", "Calcium impairs levothyroxine absorption. Take 4 hrs apart."),
    ("Levothyroxine", "Iron",            "moderate", "Iron impairs levothyroxine absorption. Take 4 hrs apart."),
    ("Alprazolam",    "Alcohol",         "high",     "CNS depression potentiation. Avoid."),
    ("Clonazepam",    "Alcohol",         "high",     "CNS depression. Serious respiratory depression risk."),
    ("Amitriptyline", "MAO Inhibitors",  "high",     "Serotonin syndrome risk. Contraindicated."),
    ("Escitalopram",  "MAO Inhibitors",  "high",     "Serotonin syndrome. Contraindicated — 14 day washout needed."),
    ("Metoprolol",    "Verapamil",       "high",     "Additive AV block and bradycardia. Avoid."),
    ("Telmisartan",   "Potassium",       "moderate", "ARBs raise potassium. Monitor serum K+ with supplements."),
    ("Ramipril",      "Potassium",       "moderate", "ACE inhibitors raise potassium. Monitor serum K+."),
    ("Ibuprofen",     "Ramipril",        "moderate", "NSAIDs reduce ACE inhibitor antihypertensive effect."),
    ("Ibuprofen",     "Telmisartan",     "moderate", "NSAIDs reduce ARB antihypertensive effect."),
    ("Fluconazole",   "Warfarin",        "high",     "Fluconazole strongly inhibits CYP2C9 — raises warfarin levels."),
    ("Pantoprazole",  "Clopidogrel",     "moderate", "PPIs reduce clopidogrel activation. Use alternative PPI if needed."),
    ("Prednisolone",  "Ibuprofen",       "high",     "Corticosteroid + NSAID — high GI bleed risk. Avoid."),
    ("Prednisolone",  "Metformin",       "moderate", "Steroids raise blood sugar — adjust antidiabetic dose."),
    ("Paracetamol",   "Alcohol",         "moderate", "Chronic heavy alcohol use + paracetamol → hepatotoxicity."),
]

DOSAGE_RANGES = [
    ("Paracetamol",       500,   1000,  "mg",   "Every 4-6 hrs, max 4g/day. Reduce in liver disease."),
    ("Ibuprofen",         200,   800,   "mg",   "Every 6-8 hrs with food. Max 2400mg/day OTC."),
    ("Metformin",         500,   2000,  "mg",   "Daily (divided doses). Start low, titrate up."),
    ("Atorvastatin",      10,    80,    "mg",   "Once daily, any time. Higher doses for high CV risk."),
    ("Rosuvastatin",      5,     40,    "mg",   "Once daily. Max 20mg with certain drug interactions."),
    ("Amlodipine",        2.5,   10,    "mg",   "Once daily. Start 5mg; max 10mg."),
    ("Telmisartan",       20,    80,    "mg",   "Once daily. Usual maintenance 40-80mg."),
    ("Ramipril",          1.25,  10,    "mg",   "Once or twice daily. Titrate over weeks."),
    ("Metoprolol",        25,    200,   "mg",   "Daily (divided). Titrate to heart rate response."),
    ("Atenolol",          25,    100,   "mg",   "Once daily. Reduce dose in renal impairment."),
    ("Levothyroxine",     25,    200,   "mcg",  "Once daily on empty stomach, 30-60 min before food."),
    ("Aspirin",           75,    325,   "mg",   "75-100mg once daily for antiplatelet. 300-600mg for pain."),
    ("Clopidogrel",       75,    75,    "mg",   "75mg once daily maintenance."),
    ("Azithromycin",      250,   500,   "mg",   "Once daily for 3-5 days depending on indication."),
    ("Ciprofloxacin",     250,   750,   "mg",   "Twice daily. Duration varies by infection type."),
    ("Levofloxacin",      250,   750,   "mg",   "Once daily. Reduce in renal impairment."),
    ("Omeprazole",        20,    40,    "mg",   "Once or twice daily before meals."),
    ("Pantoprazole",      20,    40,    "mg",   "Once daily before breakfast."),
    ("Pregabalin",        75,    300,   "mg",   "Twice daily. Start low, titrate to response."),
    ("Gabapentin",        300,   3600,  "mg",   "Divided 3 times daily. Titrate slowly."),
    ("Amitriptyline",     10,    150,   "mg",   "Start 10-25mg at night. Titrate for depression/pain."),
    ("Escitalopram",      5,     20,    "mg",   "Once daily. Start 5-10mg; usual dose 10-20mg."),
    ("Alprazolam",        0.25,  4,     "mg",   "2-3 times daily. Use lowest effective dose."),
    ("Clonazepam",        0.25,  20,    "mg",   "Divided doses. Epilepsy doses higher than anxiety."),
    ("Fluconazole",       50,    400,   "mg",   "Once daily. Duration depends on infection type."),
    ("Itraconazole",      100,   400,   "mg",   "Once or twice daily with food."),
    ("Tamsulosin",        0.4,   0.8,   "mg",   "Once daily after meal. Usual dose 0.4mg."),
    ("Montelukast",       4,     10,    "mg",   "Once daily in evening. 4mg for <6 years, 10mg adult."),
    ("Fexofenadine",      60,    180,   "mg",   "Once or twice daily. 180mg once daily common."),
    ("Cetirizine",        5,     10,    "mg",   "Once daily at night. Reduce in renal impairment."),
]

print(f"✅ Interaction data: {len(INTERACTIONS)} pairs")
print(f"✅ Dosage data: {len(DOSAGE_RANGES)} drugs")

In [ ]:
def build_sqlite_db(drugs: list[dict]):
    print("🗄️  Building SQLite database...")

    conn = sqlite3.connect(DB_PATH)
    cur  = conn.cursor()

    # ── drugs table ──────────────────────────────────────
    cur.execute("DROP TABLE IF EXISTS drugs")
    cur.execute("""
        CREATE TABLE drugs (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            brand_name   TEXT,
            generic_name TEXT NOT NULL,
            category     TEXT,
            source       TEXT
        )
    """)
    for d in drugs:
        cur.execute(
            "INSERT INTO drugs (brand_name, generic_name, category, source) VALUES (?,?,?,?)",
            (d["brand_name"], d["generic_name"], d["category"], d["source"])
        )

    # ── interactions table ────────────────────────────────
    cur.execute("DROP TABLE IF EXISTS interactions")
    cur.execute("""
        CREATE TABLE interactions (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            drug_a      TEXT NOT NULL,
            drug_b      TEXT NOT NULL,
            severity    TEXT CHECK(severity IN ('low','moderate','high')),
            description TEXT
        )
    """)
    for drug_a, drug_b, severity, desc in INTERACTIONS:
        cur.execute(
            "INSERT INTO interactions VALUES (NULL,?,?,?,?)",
            (drug_a, drug_b, severity, desc)
        )
        # Insert reverse pair too so lookup works both ways
        cur.execute(
            "INSERT INTO interactions VALUES (NULL,?,?,?,?)",
            (drug_b, drug_a, severity, desc)
        )

    # ── dosage_ranges table ───────────────────────────────
    cur.execute("DROP TABLE IF EXISTS dosage_ranges")
    cur.execute("""
        CREATE TABLE dosage_ranges (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            generic_name TEXT NOT NULL,
            min_dose     REAL,
            max_dose     REAL,
            unit         TEXT,
            notes        TEXT
        )
    """)
    for generic, mn, mx, unit, notes in DOSAGE_RANGES:
        cur.execute(
            "INSERT INTO dosage_ranges VALUES (NULL,?,?,?,?,?)",
            (generic, mn, mx, unit, notes)
        )

    conn.commit()
    conn.close()

    print(f"✅ SQLite database '{DB_PATH}' built")
    print(f"   • drugs table         : {len(drugs)} records")
    print(f"   • interactions table  : {len(INTERACTIONS) * 2} records (bidirectional)")
    print(f"   • dosage_ranges table : {len(DOSAGE_RANGES)} records\n")


build_sqlite_db(all_drugs)

## Step 8 — Verify the Knowledge Base

Quick sanity checks to confirm everything built correctly.

In [ ]:
# ── Verify Qdrant ─────────────────────────────
client = QdrantClient(path=QDRANT_PATH)
info   = client.get_collection(COLLECTION_NAME)
print("📊 Qdrant Collection Info:")
print(f"   Collection : {COLLECTION_NAME}")
print(f"   Vectors    : {info.vectors_count}")
print(f"   Status     : {info.status}")

# ── Verify SQLite ─────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
print("\n📊 SQLite Table Counts:")
for table in ["drugs", "interactions", "dosage_ranges"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"   {table:20s}: {cur.fetchone()[0]} rows")

print("\n🔍 Sample — Warfarin + Aspirin interaction:")
cur.execute("SELECT severity, description FROM interactions WHERE drug_a='Warfarin' AND drug_b='Aspirin'")
row = cur.fetchone()
if row:
    print(f"   Severity    : {row[0]}")
    print(f"   Description : {row[1]}")

print("\n🔍 Sample — Metformin dosage range:")
cur.execute("SELECT min_dose, max_dose, unit, notes FROM dosage_ranges WHERE generic_name='Metformin'")
row = cur.fetchone()
if row:
    print(f"   Range : {row[0]}-{row[1]} {row[2]}")
    print(f"   Notes : {row[3]}")
conn.close()

## ✅ Knowledge Base Build Complete!

| Output | Location | Used for |
|--------|----------|----------|
| Qdrant vector store | `./qdrant_data/` | Hybrid RAG drug lookups |
| SQLite database | `prescribot.db` | Interaction + dosage checks |
| BM25 vocabulary | `bm25_vocab.json` | Sparse vector retrieval |

**Next step:** see app.py, extraction.py, knowledge_base.py, explain.py, and sarvam_translator.py in this project for the rest of the pipeline (Claude vision + Sarvam translation, no Groq).